# 07 — K-Means Student Clustering

**Project:** MARS — Multi-Agent Recommender System for Personalized Learning  
**Agent:** PersonalizationAgent  
**Purpose:** Cluster students into learner archetypes using K-Means on
8 behavioural features.  Select optimal K via Silhouette Score and
generate paper figures (elbow, t-SNE, radar, box plots).

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8-paper")
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300,
    "font.size": 11, "axes.titlesize": 13,
    "axes.labelsize": 12, "figure.figsize": (8, 5),
    "savefig.bbox": "tight",
})

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

import logging
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

print("Libraries loaded.")

## 1. Load Data & Extract Features

In [ ]:
from data.loader import EdNetLoader
from agents.personalization_agent import (
    PersonalizationAgent, extract_user_features,
    FEATURE_NAMES, FEATURE_NAMES_SCALAR, FEATURE_NAMES_ONEHOT,
    ClusterProfile, K_RANGE,
)

loader = EdNetLoader(data_dir="../data/raw")
interactions = loader.load_interactions(sample_users=1000)
print(f"Interactions: {len(interactions):,} ({interactions['user_id'].nunique()} users)")

# Extract features
user_features = extract_user_features(interactions)
print(f"\nUser features shape: {user_features.shape}")
print(f"Features: {FEATURE_NAMES}")
user_features.describe().round(3)

In [ ]:
# Feature distributions
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
colors = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12",
          "#9b59b6", "#e67e22", "#1abc9c"]

for i, feat in enumerate(FEATURE_NAMES_SCALAR):
    ax = axes[i // 4, i % 4]
    ax.hist(user_features[feat].dropna(), bins=30,
            color=colors[i % len(colors)], edgecolor="black",
            linewidth=0.3, alpha=0.8)
    ax.set_title(feat, fontsize=10)
    ax.set_ylabel("Count")
    sns.despine(ax=ax)

# Last subplot: dominant part distribution
ax = axes[1, 3]
dom_part = user_features[FEATURE_NAMES_ONEHOT].idxmax(axis=1).str.replace("dominant_part_", "Part ")
dom_part.value_counts().sort_index().plot.bar(
    ax=ax, color="#34495e", edgecolor="black", linewidth=0.3)
ax.set_title("Dominant Part", fontsize=10)
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=0)
sns.despine(ax=ax)

fig.suptitle("Feature Distributions (per user)", fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_cluster_feature_distributions.png")
plt.show()

## 2. Train Clusters (Silhouette Selection)

In [ ]:
agent = PersonalizationAgent()
optimal_k = agent.train_clusters(user_features)

print(f"\nOptimal K: {optimal_k}")
print(f"Silhouette scores: {agent.silhouette_scores}")
print(f"Inertias: {agent.inertias}")
print(f"Cluster names: {agent._cluster_names}")

## 3. Elbow Plot + Silhouette Score Plot

In [ ]:
ks = sorted(agent.silhouette_scores.keys())
sils = [agent.silhouette_scores[k] for k in ks]
inertias = [agent.inertias[k] for k in ks]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Elbow plot
ax = axes[0]
ax.plot(ks, inertias, "o-", color="#3498db", markersize=8, linewidth=2)
ax.axvline(optimal_k, color="red", linestyle="--", alpha=0.6,
           label=f"Optimal K={optimal_k}")
ax.set_xlabel("Number of Clusters (K)")
ax.set_ylabel("Inertia (Within-Cluster Sum of Squares)")
ax.set_title("Elbow Method")
ax.set_xticks(ks)
ax.legend()
sns.despine(ax=ax)

# Right: Silhouette
ax = axes[1]
bars = ax.bar(ks, sils, color="#2ecc71", edgecolor="black", linewidth=0.5)
# Highlight optimal
bars[ks.index(optimal_k)].set_color("#e74c3c")
for bar, val in zip(bars, sils):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
            f"{val:.3f}", ha="center", fontsize=9)
ax.set_xlabel("Number of Clusters (K)")
ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette Score by K")
ax.set_xticks(ks)
ax.set_ylim(0, max(sils) * 1.2)
sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_cluster_elbow_silhouette.png")
plt.show()

## 4. t-SNE Visualization

In [ ]:
# Scale features for t-SNE
X_scaled = agent.scaler.transform(
    np.nan_to_num(user_features[FEATURE_NAMES].values, nan=0.0)
)

# Get cluster labels for all users
labels = agent.model.predict(X_scaled)

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_2d = tsne.fit_transform(X_scaled)

# Try UMAP if available
try:
    import umap
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15)
    X_umap = reducer.fit_transform(X_scaled)
    has_umap = True
except ImportError:
    has_umap = False
    print("UMAP not available, using t-SNE only")

n_plots = 2 if has_umap else 1
fig, axes = plt.subplots(1, n_plots, figsize=(8 * n_plots, 7))
if n_plots == 1:
    axes = [axes]

cluster_colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12",
                  "#9b59b6", "#e67e22", "#1abc9c", "#34495e"]

# t-SNE plot
ax = axes[0]
for cid in range(optimal_k):
    mask = labels == cid
    name = agent._cluster_names.get(cid, f"Cluster {cid}")
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
              c=cluster_colors[cid % len(cluster_colors)],
              s=15, alpha=0.6, label=f"{name} (n={mask.sum()})")
ax.set_title("t-SNE Visualization of Student Clusters")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(fontsize=9, markerscale=2)
sns.despine(ax=ax)

# UMAP plot
if has_umap:
    ax = axes[1]
    for cid in range(optimal_k):
        mask = labels == cid
        name = agent._cluster_names.get(cid, f"Cluster {cid}")
        ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
                  c=cluster_colors[cid % len(cluster_colors)],
                  s=15, alpha=0.6, label=f"{name} (n={mask.sum()})")
    ax.set_title("UMAP Visualization of Student Clusters")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.legend(fontsize=9, markerscale=2)
    sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_cluster_tsne.png")
plt.show()

## 5. Radar Chart: Mean Profile per Cluster

In [ ]:
# Get centroids in original scale
centroids_df = agent.get_centroids_df()
print("Cluster centroids (original scale):")
centroids_df.round(3)

In [ ]:
# Radar chart with scalar features (7 axes)
from matplotlib.patches import FancyBboxPatch

radar_features = FEATURE_NAMES_SCALAR
n_features = len(radar_features)
angles = np.linspace(0, 2 * np.pi, n_features, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

# Normalise centroids to [0, 1] for radar
centroid_values = centroids_df[radar_features].values
mins = centroid_values.min(axis=0)
maxs = centroid_values.max(axis=0)
ranges = maxs - mins
ranges[ranges == 0] = 1.0
normed = (centroid_values - mins) / ranges

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

for cid in range(optimal_k):
    values = normed[cid].tolist()
    values += values[:1]  # close
    name = agent._cluster_names.get(cid, f"Cluster {cid}")
    color = cluster_colors[cid % len(cluster_colors)]
    ax.plot(angles, values, "o-", color=color, linewidth=2,
            markersize=5, label=name)
    ax.fill(angles, values, color=color, alpha=0.1)

# Labels
ax.set_xticks(angles[:-1])
short_names = [
    "Avg Time", "Accuracy", "Changed Rate",
    "Sessions/wk", "Q/Session", "False Conf", "Learn Speed",
]
ax.set_xticklabels(short_names, fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_title("Cluster Profiles (Radar Chart)", fontsize=14, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)

fig.savefig(RESULTS_DIR / "fig_cluster_radar.png")
plt.show()

## 6. Cluster Characteristics Table

In [ ]:
summary = agent.cluster_summary()
print("\nCluster Summary:")
print(summary.to_string(index=False))

# Save for paper
summary.to_csv(RESULTS_DIR / "cluster_summary.csv", index=False)

# Styled table as figure
fig, ax = plt.subplots(figsize=(16, 2 + 0.5 * len(summary)))
ax.axis("off")

# Prepare display columns
disp = summary[[
    "cluster_name", "size", "pct",
    "mean_accuracy_rate", "mean_avg_elapsed_time",
    "mean_learning_speed", "mean_false_confidence_rate",
    "mean_session_frequency",
]].copy()
disp.columns = [
    "Cluster", "Size", "%",
    "Accuracy", "Avg Time",
    "Learn Speed", "False Conf",
    "Sess/wk",
]

table = ax.table(
    cellText=disp.values,
    colLabels=disp.columns,
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)

# Color header
for j in range(len(disp.columns)):
    table[0, j].set_facecolor("#34495e")
    table[0, j].set_text_props(color="white", fontweight="bold")

# Color cluster rows
for i in range(len(disp)):
    for j in range(len(disp.columns)):
        table[i + 1, j].set_facecolor(
            cluster_colors[i % len(cluster_colors)] + "30"  # with alpha
        )

ax.set_title("Cluster Characteristics", fontsize=14, pad=20)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_cluster_table.png")
plt.show()

## 7. Accuracy Distribution by Cluster (Box Plot)

In [ ]:
# Build DataFrame with cluster labels
plot_df = user_features[FEATURE_NAMES_SCALAR].copy()
plot_df["cluster"] = labels
plot_df["cluster_name"] = [
    agent._cluster_names.get(c, f"Cluster {c}") for c in labels
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Box plot: accuracy by cluster
ax = axes[0]
order = sorted(plot_df["cluster_name"].unique())
palette = {name: cluster_colors[i % len(cluster_colors)]
           for i, name in enumerate(order)}
sns.boxplot(data=plot_df, x="cluster_name", y="accuracy_rate",
            order=order, palette=palette, ax=ax,
            linewidth=0.8, fliersize=3)
ax.set_xlabel("Cluster")
ax.set_ylabel("Accuracy Rate")
ax.set_title("Accuracy Distribution by Cluster")
ax.tick_params(axis="x", rotation=25)
sns.despine(ax=ax)

# Box plot: avg elapsed time by cluster
ax = axes[1]
sns.boxplot(data=plot_df, x="cluster_name", y="avg_elapsed_time",
            order=order, palette=palette, ax=ax,
            linewidth=0.8, fliersize=3)
ax.set_xlabel("Cluster")
ax.set_ylabel("Avg Elapsed Time (normalised)")
ax.set_title("Response Time by Cluster")
ax.tick_params(axis="x", rotation=25)
sns.despine(ax=ax)

# Box plot: learning speed by cluster
ax = axes[2]
sns.boxplot(data=plot_df, x="cluster_name", y="learning_speed",
            order=order, palette=palette, ax=ax,
            linewidth=0.8, fliersize=3)
ax.set_xlabel("Cluster")
ax.set_ylabel("Learning Speed (slope)")
ax.set_title("Learning Speed by Cluster")
ax.axhline(0, color="gray", linestyle=":", alpha=0.5)
ax.tick_params(axis="x", rotation=25)
sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_cluster_boxplots.png")
plt.show()

## 8. Feature Correlation Heatmap (per cluster)

In [ ]:
# Overall correlation
fig, ax = plt.subplots(figsize=(9, 7))
corr = user_features[FEATURE_NAMES_SCALAR].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1)
ax.set_title("Feature Correlation Matrix")

fig.savefig(RESULTS_DIR / "fig_cluster_feature_correlation.png")
plt.show()

## 9. Agent Integration Test

In [ ]:
# Test assign_cluster
test_user = user_features.index[0]
profile = agent.assign_cluster(str(test_user))
print(f"User: {test_user}")
print(f"Cluster: {profile['cluster_name']}")
print(f"User type: {profile['user_type']}")
print(f"Distance to centroid: {profile['distance_to_centroid']}")

# Test get_user_type
user_type = agent.get_user_type(str(test_user))
print(f"\nget_user_type: {user_type}")

# Test personalize
pers_result = agent.personalize(str(test_user))
print(f"\npersonalize result:")
print(f"  Cluster: {pers_result['cluster_name']}")
print(f"  Adjustments: {pers_result['adjustments']}")

# Test cold start
cold_result = agent.assign_cluster(
    user_id="new_user_999",
    diagnostic={
        "responses": [
            {"correct": True, "part_id": 1},
            {"correct": True, "part_id": 2},
            {"correct": False, "part_id": 3},
        ],
    },
    confidence={
        "class_names": ["SOLID", "UNSURE_CORRECT", "FALSE_CONFIDENCE"],
    },
)
print(f"\nCold start user: {cold_result['cluster_name']}")

## 10. Summary for Paper

In [ ]:
print("\n" + "=" * 55)
print("  K-Means Clustering — Summary for Paper")
print("=" * 55)
print(f"  Optimal K:              {optimal_k}")
print(f"  Silhouette Score:       {agent.silhouette_scores[optimal_k]:.4f}")
print(f"  Features:               {len(FEATURE_NAMES)} ({len(FEATURE_NAMES_SCALAR)} scalar + 7 one-hot)")
print(f"  Users clustered:        {len(user_features)}")
print()
for cid in range(optimal_k):
    n = sum(1 for v in agent._user_clusters.values() if v == cid)
    name = agent._cluster_names.get(cid, f"Cluster {cid}")
    print(f"  Cluster {cid} ({name:20s}): n={n}")

print(f"\n  Figures saved to: {RESULTS_DIR.resolve()}")
print("=" * 55)